# Qwen Model Inference Notebook

This notebook loads and runs the Qwen 3.5-4B model from local files.

## 1. Import Required Libraries

Import necessary libraries including transformers, torch, os, and pathlib for model loading and file path handling.

In [8]:
import os
import sys
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from pathlib import Path
import sentencepiece

## 2. Set Model File Path

Define the model file path using absolute or relative paths. Use os.path or pathlib to construct the correct path to the Qwen model directory.

In [ ]:
# Resolve project root dynamically from the current notebook location
project_root = Path.cwd()
if not (project_root / "backend").exists() and (project_root.parent / "backend").exists():
    project_root = project_root.parent

BASE_DIR = str(project_root)

# Add backend to Python path once
backend_path = os.path.join(BASE_DIR, "backend")
if backend_path not in sys.path:
    sys.path.append(backend_path)

# Prefer a local Instruct snapshot if present, otherwise use Hub model
local_instruct_snapshots = sorted(
    (project_root / "models" / "qwen" / "models--Qwen--Qwen3.5-4B-Instruct" / "snapshots").glob("*"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

if local_instruct_snapshots:
    MODEL_LOCAL_PATH = str(local_instruct_snapshots[0])
    print(f"Using local model snapshot: {MODEL_LOCAL_PATH}")
else:
    MODEL_LOCAL_PATH = "Qwen/Qwen3.5-4B-Instruct"
    print(f"Local Instruct snapshot not found. Falling back to: {MODEL_LOCAL_PATH}")

Model path found: /home/yesoytur/APilus/models/qwen/models--Qwen--Qwen3.5-4B/snapshots/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a
Model files: ['chat_template.jinja', 'config.json', 'tokenizer_config.json', 'model.safetensors.index.json', 'model.safetensors-00001-of-00002.safetensors', 'model.safetensors-00002-of-00002.safetensors', 'tokenizer.json']


## 3. Load the Qwen Model

Use the transformers library to load the Qwen model from the specified file path using AutoModelForCausalLM and AutoTokenizer.

In [ ]:
print(f"Loading Qwen model from: {MODEL_LOCAL_PATH}")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_LOCAL_PATH,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

use_cuda = torch.cuda.is_available()
if use_cuda and torch.cuda.is_bf16_supported():
    model_dtype = torch.bfloat16
elif use_cuda:
    model_dtype = torch.float16
else:
    model_dtype = torch.float32

model = AutoModelForCausalLM.from_pretrained(
    MODEL_LOCAL_PATH,
    torch_dtype=model_dtype,
    device_map="auto" if use_cuda else None,
    trust_remote_code=True,
)

# Configure model
model.config.pad_token_id = tokenizer.pad_token_id
model.eval()

print(f"Model loaded successfully (dtype={model_dtype}, cuda={use_cuda})")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading Qwen Model from: /home/yesoytur/APilus/models/qwen/models--Qwen--Qwen3.5-4B/snapshots/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


Model successfully loaded!


## 4. Run Inference

Execute inference on sample text inputs using the loaded model and tokenizer to generate predictions.

In [ ]:
import torch

def _safe_token_id(token_text):
    vocab = tokenizer.get_vocab()
    return tokenizer.convert_tokens_to_ids(token_text) if token_text in vocab else None


def generate_answer(user_prompt):
    messages = [
        {
            "role": "system",
            "content": "You are a helpful conversational assistant. Answer clearly and directly.",
        },
        {"role": "user", "content": user_prompt},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    eos_ids = [tokenizer.eos_token_id]
    for stop_token in ["<|im_end|>", "<|eot_id|>", "<|end_of_text|>"]:
        stop_id = _safe_token_id(stop_token)
        if stop_id is not None:
            eos_ids.append(stop_id)

    # Prevent common tool/FIM artifact tokens from being generated.
    blocked_tokens = [
        "<|fim_prefix|>",
        "<|fim_middle|>",
        "<|fim_suffix|>",
        "<|fim_pad|>",
        "<|file_sep|>",
        "<|repo_name|>",
        "<tool_call>",
        "</tool_call>",
        "<tool_response>",
        "</tool_response>",
        "<think>",
        "</think>",
    ]
    bad_words_ids = []
    for tok in blocked_tokens:
        tok_id = _safe_token_id(tok)
        if tok_id is not None:
            bad_words_ids.append([tok_id])

    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=256,
            do_sample=False,
            temperature=None,
            top_p=None,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=list(dict.fromkeys(eos_ids)),
            bad_words_ids=bad_words_ids if bad_words_ids else None,
        )

    input_len = model_inputs.input_ids.shape[1]
    new_tokens = generated_ids[:, input_len:]

    response = tokenizer.decode(
        new_tokens[0],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    ).strip()

    return response if response else "I could not generate a valid response."

# Test prompt
user_prompt = "Hello, can you tell me about yourself?"
response = generate_answer(user_prompt)
print(f"User: {user_prompt}")
print(f"\nAssistant: {response}")

User: Hello, can you tell me about yourself?

Assistant: I'm having trouble generating a response. Please try again.


## 5. Additional Examples

Try different prompts to test the model.

In [14]:
# Example 1: Simple question
prompt1 = "What is the capital of France?"
response1 = generate_answer(prompt1)
print(f"Q: {prompt1}")
print(f"A: {response1}\n")

# Example 2: More complex question
prompt2 = "Explain quantum computing in simple terms."
response2 = generate_answer(prompt2)
print(f"Q: {prompt2}")
print(f"A: {response2}\n")

# You can add your own prompts here
# prompt3 = "Your custom prompt here"
# response3 = generate_answer(prompt3)
# print(f"Q: {prompt3}")
# print(f"A: {response3}")

Q: What is the capital of France?
A: </tool_call><|fim_prefix|><tool_response><|fim_middle|></tool_response></tool_response><|fim_pad|><|fim_suffix|><|repo_name|><|fim_pad|><|file_sep|><think><|fim_suffix|><|fim_suffix|><|fim_suffix|><|fim_suffix|><|repo_name|><think></tool_call><|fim_prefix|><tool_response><|fim_middle|>

Q: Explain quantum computing in simple terms.
A: <|fim_pad|><|repo_name|><|repo_name|><|fim_prefix|><|fim_middle|><tool_call><|fim_suffix|><|fim_prefix|></tool_call><|fim_middle|><|fim_suffix|><|file_sep|></tool_response><tool_response><|fim_suffix|></tool_response>



## 6. Tokens Per Second Benchmark

Measure generation throughput (tokens/second) for one prompt using the currently loaded model and tokenizer.

In [ ]:
import time
import torch

def benchmark_tokens_per_second(prompt="Explain quantum computing in simple terms.", max_new_tokens=128):
    messages = [
        {
            "role": "system",
            "content": "You are a helpful conversational assistant. Answer clearly and directly.",
        },
        {"role": "user", "content": prompt},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    input_len = model_inputs.input_ids.shape[1]

    # Warmup run for more stable timing.
    with torch.no_grad():
        _ = model.generate(
            **model_inputs,
            max_new_tokens=16,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start = time.perf_counter()
    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    new_tokens = generated_ids.shape[1] - input_len
    tps = new_tokens / elapsed if elapsed > 0 else float("nan")

    print(f"Prompt tokens: {input_len}")
    print(f"Generated tokens: {new_tokens}")
    print(f"Elapsed time: {elapsed:.3f} sec")
    print(f"Tokens per second: {tps:.2f}")

# Run benchmark
benchmark_tokens_per_second()